# 02 — Literacy & Education
**Source:** India Districts Census 2011 · 640 districts  
Covers literacy rates, gender literacy gap, and educational attainment levels.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.male, f.female,
           f.literate, f.male_literate, f.female_literate,
           f.below_primary_education, f.primary_education,
           f.middle_education, f.secondary_education,
           f.higher_education, f.graduate_education,
           f.illiterate_education, f.total_education,
           f.power_parity_less_than_rs_45000,
           f.power_parity_rs_45000_90000,
           f.power_parity_rs_90000_150000,
           f.power_parity_rs_150000_240000,
           f.power_parity_rs_240000_330000,
           f.power_parity_rs_330000_425000,
           f.power_parity_rs_425000_545000,
           f.power_parity_above_rs_545000,
           f.total_power_parity
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df['literacy_rate']        = (df['literate'] / df['population'] * 100).round(2)
df['male_literacy_rate']   = (df['male_literate'] / df['male'] * 100).round(2)
df['female_literacy_rate'] = (df['female_literate'] / df['female'] * 100).round(2)
df['gender_literacy_gap']  = (df['male_literacy_rate'] - df['female_literacy_rate']).round(2)

state = df.groupby('state_name').agg(
    population=('population','sum'), literate=('literate','sum'),
    male=('male','sum'), female=('female','sum'),
    male_literate=('male_literate','sum'),
    female_literate=('female_literate','sum'),
    below_primary=('below_primary_education','sum'),
    primary=('primary_education','sum'),
    middle=('middle_education','sum'),
    secondary=('secondary_education','sum'),
    higher=('higher_education','sum'),
    graduate=('graduate_education','sum'),
    total_edu=('total_education','sum'),
).reset_index()

state['literacy_rate']        = (state['literate'] / state['population'] * 100).round(2)
state['male_literacy_rate']   = (state['male_literate'] / state['male'] * 100).round(2)
state['female_literacy_rate'] = (state['female_literate'] / state['female'] * 100).round(2)
state['gender_gap']           = (state['male_literacy_rate'] - state['female_literacy_rate']).round(2)

print(f'National Avg Literacy: {df.literate.sum()/df.population.sum()*100:.2f}%')

## 2.1 — Literacy Rate by State (Male vs Female)

In [ ]:
state_sorted = state.sort_values('female_literacy_rate')

fig = go.Figure()
fig.add_trace(go.Bar(name='Male Literacy %',   y=state_sorted['state_name'], x=state_sorted['male_literacy_rate'],   orientation='h', marker_color='#3498db'))
fig.add_trace(go.Bar(name='Female Literacy %', y=state_sorted['state_name'], x=state_sorted['female_literacy_rate'], orientation='h', marker_color='#e91e63'))

fig.update_layout(
    barmode='group',
    title='Male vs Female Literacy Rate by State — sorted by Female Literacy',
    xaxis_title='Literacy Rate (%)',
    height=700,
    legend=dict(orientation='h', y=1.02)
)
fig.show()

## 2.2 — Gender Literacy Gap by State

In [ ]:
gap_sorted = state.sort_values('gender_gap', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if g > 15 else '#f39c12' if g > 10 else '#2ecc71' for g in gap_sorted['gender_gap']]
ax.barh(gap_sorted['state_name'], gap_sorted['gender_gap'], color=colors)
ax.axvline(state['gender_gap'].mean(), color='navy', linestyle='--', label=f"National avg gap: {state['gender_gap'].mean():.1f}pp")
ax.set_xlabel('Gender Literacy Gap (Male% − Female%)')
ax.set_title('Gender Literacy Gap by State\n(Red > 15pp | Orange 10–15pp | Green ≤ 10pp)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 2.3 — Education Attainment Funnel (National)

In [ ]:
edu_labels = ['Below Primary', 'Primary', 'Middle', 'Secondary', 'Higher Secondary', 'Graduate+']
edu_cols   = ['below_primary', 'primary', 'middle', 'secondary', 'higher', 'graduate']
edu_vals   = [state[c].sum() for c in edu_cols]
edu_pct    = [v / sum(edu_vals) * 100 for v in edu_vals]

fig = go.Figure(go.Funnel(
    y=edu_labels, x=edu_vals,
    texttemplate='%{label}<br>%{value:,.0f} (%{percentInitial:.1%})',
    marker_color=['#1a237e','#1565c0','#1976d2','#42a5f5','#90caf9','#bbdefb'],
))
fig.update_layout(title='Educational Attainment Funnel — National Level (2011 Census)', height=500)
fig.show()

## 2.4 — Education Attainment by State (Stacked %)

In [ ]:
edu_pcts = state.copy()
for col, label in zip(edu_cols, edu_labels):
    edu_pcts[col+'_pct'] = edu_pcts[col] / edu_pcts['total_edu'] * 100

edu_pcts = edu_pcts.sort_values('graduate_pct', ascending=False)

fig = go.Figure()
palette = ['#1a237e','#1565c0','#1976d2','#42a5f5','#90caf9','#bbdefb']
for col, label, color in zip([c+'_pct' for c in edu_cols], edu_labels, palette):
    fig.add_trace(go.Bar(name=label, x=edu_pcts['state_name'], y=edu_pcts[col], marker_color=color))

fig.update_layout(
    barmode='stack',
    title='Education Attainment Distribution by State (%) — sorted by Graduate Share',
    yaxis_title='% of Educated Population',
    xaxis_tickangle=-45,
    height=550,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 2.5 — Literacy Rate vs Income (Power Parity)

In [ ]:
income = df.copy()
income = income[income['total_power_parity'] > 0]
income['high_income_pct'] = (income['power_parity_above_rs_545000'] / income['total_power_parity'] * 100).round(2)
income['low_income_pct']  = (income['power_parity_less_than_rs_45000'] / income['total_power_parity'] * 100).round(2)

fig = px.scatter(
    income,
    x='literacy_rate', y='high_income_pct',
    color='state_name',
    hover_name='district_name',
    hover_data={'state_name': True, 'literacy_rate': ':.1f', 'high_income_pct': ':.2f'},
    labels={'literacy_rate': 'Literacy Rate (%)', 'high_income_pct': 'High Income HH % (>₹5.45L)'},
    title='Literacy Rate vs High-Income Household Share — District Level',
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='black',
)
fig.update_layout(showlegend=False, height=550)
fig.show()

## 2.6 — Top & Bottom 10 Districts by Literacy Rate

In [ ]:
top10    = df.nlargest(10, 'literacy_rate')[['district_name','state_name','literacy_rate']]
bottom10 = df.nsmallest(10, 'literacy_rate')[['district_name','state_name','literacy_rate']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top10['district_name'] + ', ' + top10['state_name'], top10['literacy_rate'],
         color=sns.color_palette('Greens_r', 10))
ax1.set_xlabel('Literacy Rate (%)')
ax1.set_title('Top 10 Most Literate Districts', fontweight='bold')
ax1.invert_yaxis()
ax1.set_xlim(60, 102)

ax2.barh(bottom10['district_name'] + ', ' + bottom10['state_name'], bottom10['literacy_rate'],
         color=sns.color_palette('Reds_r', 10))
ax2.set_xlabel('Literacy Rate (%)')
ax2.set_title('Bottom 10 Least Literate Districts', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 2.7 — District Literacy Rate Distribution (Box Plot by State)

In [ ]:
state_order = df.groupby('state_name')['literacy_rate'].median().sort_values().index.tolist()

fig = px.box(
    df, x='state_name', y='literacy_rate',
    color='state_name',
    category_orders={'state_name': state_order},
    title='Literacy Rate Distribution by State (district-level spread)',
    labels={'literacy_rate': 'Literacy Rate (%)', 'state_name': 'State'},
    points='outliers'
)
fig.update_layout(showlegend=False, xaxis_tickangle=-45, height=550)
fig.show()